# Calculate the Enhanced Combined Drought Index

In [1]:
import pandas as pd
import os
import json
import numpy as np
import rioxarray
import xarray as xr
from pyarrow.parquet import read_table as pq_read_table

## Load the drought indices data

In [2]:
PARQUET_META_KEY = b"xarray.metadata"
output_dir = "results"

In [3]:
VDI_path = os.path.join(output_dir, "VDI.parquet")
TDI_path = os.path.join(output_dir, "TDI.parquet")
PDI_path = os.path.join(output_dir, "PDI.parquet")

In [4]:
def read_table(path: str) -> pd.DataFrame:
    """Read a Parquet file with Conflux metadata.

    Arguments
    ---------
    path : str
        Path to Parquet file.

    Returns
    -------
    pd.DataFrame
        DataFrame with attrs set.
    """
    table = pq_read_table(path)
    df = table.to_pandas()
    meta_json = table.schema.metadata[PARQUET_META_KEY]
    metadata = json.loads(meta_json)
    for key, val in metadata.items():
        df.attrs[key] = val
    return df

In [5]:
VDI_df = read_table(VDI_path)
VDI = VDI_df.to_xarray()
VDI = VDI.rio.write_crs(VDI_df.attrs["crs"], inplace=True)
VDI["spatial_ref"] = VDI.rio.crs.to_epsg()
VDI

<xarray.Dataset>
Dimensions:      (dekad: 374, y: 49, x: 20)
Coordinates:
  * dekad        (dekad) datetime64[ns] 2014-01-01 2014-01-11 ... 2024-05-11
  * y            (y) float64 -2.75e+04 -2.25e+04 ... 2.075e+05 2.125e+05
  * x            (x) float64 3.428e+06 3.432e+06 ... 3.518e+06 3.522e+06
    spatial_ref  int64 6933
Data variables:
    NDVI         (dekad, y, x) float64 nan 0.02662 0.006834 ... 0.4475 0.4763

In [6]:
TDI_df = read_table(TDI_path)
TDI = TDI_df.to_xarray()
TDI = TDI.rio.write_crs(TDI_df.attrs["crs"], inplace=True)
TDI["spatial_ref"] = TDI.rio.crs.to_epsg()
TDI

<xarray.Dataset>
Dimensions:              (dekad: 374, y: 49, x: 20)
Coordinates:
  * dekad                (dekad) datetime64[ns] 2014-01-01 ... 2024-05-11
  * y                    (y) float64 -2.75e+04 -2.25e+04 ... 2.075e+05 2.125e+05
  * x                    (x) float64 3.428e+06 3.432e+06 ... 3.518e+06 3.522e+06
    spatial_ref          int64 6933
Data variables:
    surface_temperature  (dekad, y, x) float64 nan 0.1852 ... 0.2009 0.4853

In [7]:
PDI_df = read_table(PDI_path)
PDI = PDI_df.to_xarray()
PDI = PDI.rio.write_crs(PDI_df.attrs["crs"], inplace=True)
PDI["spatial_ref"] = PDI.rio.crs.to_epsg()
PDI

<xarray.Dataset>
Dimensions:      (dekad: 369, y: 49, x: 20)
Coordinates:
  * dekad        (dekad) datetime64[ns] 2014-01-01 2014-01-11 ... 2024-03-21
  * y            (y) float64 -2.75e+04 -2.25e+04 ... 2.075e+05 2.125e+05
  * x            (x) float64 3.428e+06 3.432e+06 ... 3.518e+06 3.522e+06
    spatial_ref  int64 6933
Data variables:
    rainfall     (dekad, y, x) float64 0.01281 0.01056 0.01172 ... 0.3438 0.4324

In [8]:
drought_indices = dict(rainfall=PDI.rainfall, soil_moisture=None, temperature=TDI.surface_temperature, vegetation=VDI.NDVI)

## Calculate the weight for each drought index

The weight $w$ of each individual drought index is automatically calculated for every grid point (pixel) with respect to its capability to reflect the future vegetation status (NDVI). In the case of data gaps in one input dataset, the weights are automatically redistributed to other available variables.

\begin{equation}
w_{i} = \frac{\frac{lag^*_{i}}{\sum_{j=1}^{n} lag^*_{j}} + \frac{corr^*_{i}}{\sum_{j=1}^{n} corr^*_{j}}}{2}
\end{equation}

- $w$ weight for the respective drought index 

- $lag^*$ modified time lag for the respective parameter 

- $corr^*$ modified correlation coefficient for the respective parameter 

- $i$ index for the respective parameter/drought index 

- $j$ running parameter covering all parameters used for the ECDI calculation

- $n$ number of individual drought indices used for the ECDI calculation

In [9]:
reference_drought_index = drought_indices["vegetation"]

In [10]:
def get_max_value(index, da):
    return da.isel(time_lag=index)

correlation = {}
lag = {}
for i in drought_indices.keys():
    if drought_indices[i] is not None:
        comparison_drought_index = drought_indices[i]
        
        corr_list = []
        lags=[0, 10]
        for time_lag in range(lags[0], lags[1]):
            # Modify the time lag
            time_lag += abs(lags[0]) + 1
            # Pearson's correlation coefficient
            corr = xr.corr(reference_drought_index, comparison_drought_index.shift(dekad=time_lag), dim="dekad")
            modified_corr = np.abs(corr).assign_coords(time_lag=time_lag).expand_dims({"time_lag": 1})
            corr_list.append(modified_corr)

        da_corr = xr.concat(corr_list, dim="time_lag")

        # Get the maximum modified correlation value for each pixel.
        max_corr = da_corr.max(dim="time_lag")

        # For each expand_dimspixel get the the time lag at which the maximum
        # correlation value occurs.
        max_time_lag = xr.apply_ufunc(
                get_max_value,
                da_corr.argmax(dim="time_lag"),
                kwargs={"da": da_corr.time_lag},
                vectorize=True,
                dask="allowed",)
        
        correlation[i] = max_corr
        lag[i] = max_time_lag

In [11]:
assert correlation.keys() == lag.keys()

In [12]:
weights = {}
for i in drought_indices.keys():
    if drought_indices[i] is not None:
         weights[i]=((lag[i]/sum(lag.values()) + (correlation[i]/sum(correlation.values()))))/2

## Calculate the Enhanced Combined Drought Index

The weight of every individual index is multiplied by the respective individual index to calculate the ECDI. The sum of all weights per decade is one.

\begin{equation}
ECDI = \sum_{i-1}^{n}w_{i} * \text{DI}_{i}
\end{equation}

- $ECDI$ Enhanced Combined Drought Index 

- $w$ Weight for each individual drought index (e.g., rainfall)

- $\text{DI}$ Individual drought index 

- $n$ number of drought indices used to calculate the ECDI 

- $i$ running parameter covering the number of drought indices

In [13]:
ECDI = sum([weights[i]*drought_indices[i] for i in weights.keys()])
ECDI

<xarray.DataArray (y: 49, x: 20, dekad: 369)>
array([[[       nan, 0.10995068, 0.11969659, ..., 0.77382815,
         0.72897437, 0.74338981],
        [0.06181559, 0.13517628, 0.18546672, ..., 0.73634366,
         0.70077345, 0.69902005],
        [0.04501188, 0.06896588, 0.13823978, ..., 0.70331399,
         0.67944981, 0.67372978],
        ...,
        [0.08474839, 0.11493357, 0.1412643 , ..., 0.66009479,
         0.61933963, 0.59711977],
        [0.0879113 , 0.11763632, 0.15026331, ..., 0.67853658,
         0.64173614, 0.62724411],
        [0.08202504, 0.13380279, 0.16397521, ..., 0.70311141,
         0.66427366, 0.60869019]],

       [[0.10799987, 0.1371992 , 0.15710976, ..., 0.74732507,
         0.70508358, 0.7202082 ],
        [0.04388219, 0.06248749, 0.08136152, ..., 0.7703394 ,
         0.7301508 , 0.74558787],
        [       nan, 0.13935719, 0.16114062, ..., 0.69743257,
         0.66777461, 0.66623581],
...
        [0.0598947 , 0.11513166, 0.19749709, ..., 0.65864281,
         0.62737355, 0.61645091],
        [0.06339651, 0.1293215 , 0.21449722, ..., 0.65922055,
         0.62957768, 0.6320541 ],
        [0.1376844 , 0.16268894, 0.2269202 , ..., 0.61577471,
         0.57543981, 0.54201386]],

       [[0.10055714, 0.14712215, 0.19451376, ..., 0.71304482,
         0.70222298, 0.68240538],
        [0.04869958, 0.05602148, 0.06536913, ..., 0.64694526,
         0.61242086, 0.58001313],
        [0.09549458, 0.12995256, 0.16421701, ..., 0.66522598,
         0.64502541, 0.6285216 ],
        ...,
        [0.11001436, 0.12329806, 0.17966364, ..., 0.55579637,
         0.51283435, 0.47609092],
        [0.08292331, 0.12217477, 0.18700342, ..., 0.59458756,
         0.55820952, 0.53066647],
        [0.13336506, 0.13736011, 0.14082972, ..., 0.6164461 ,
         0.57033877, 0.54167175]]])
Coordinates:
  * y            (y) float64 -2.75e+04 -2.25e+04 ... 2.075e+05 2.125e+05
  * x            (x) float64 3.428e+06 3.432e+06 ... 3.518e+06 3.522e+06
  * dekad        (dekad) datetime64[ns] 2014-01-01 2014-01-11 ... 2024-03-21
    spatial_ref  int64 6933